In [ ]:
import xarray as xr

region = "lytton"

#hotz:
lat0, lat1 = 49, 59
lon0, lon1 = 360 - 125, 360 - 115  # 235–245

#Z&B:
lon0, lon1 = 360 - 123, 360 - 115  # example: 235–245
lat0, lat1 = 43, 55
# ---- Maeve ----
lon0, lon1 = 360 - 123, 360 - 118  # example: 235–245
lat0, lat1 = 48, 55

#paris:
#lon0, lon1 = 1.5, 6
#lat0, lat1 = 47.0, 51.0

path = "/work/uc1275/u301827/02_MSE/full_midlatitude/era5_midlatitudes_JJA_compiled.nc"

# open lazily (tune time chunk)
ds0 = xr.open_dataset(
    path,
    chunks={"time": 200},   # try 200–1000 depending on file + I/O
    cache=False,
)

# subset lon/lat (cheap)
lon_slice = slice(lon0, lon1) if ds0.lon[0] < ds0.lon[-1] else slice(lon1, lon0)
lat_slice = slice(lat0, lat1) if ds0.lat[0] < ds0.lat[-1] else slice(lat1, lat0)
ds_box = ds0.sel(lon=lon_slice, lat=lat_slice)

# regional mean
ds_reg = ds_box.mean(dim=["lon", "lat"], skipna=True)

# squeeze singleton dims
dims_to_remove = ["depth", "depth_2", "lev", "nhyi", "nhym"]
singleton_dims = [d for d in dims_to_remove if d in ds_reg.dims and ds_reg.sizes[d] == 1]
ds_reg = ds_reg.squeeze(singleton_dims, drop=True)

# trigger computation once
ds_reg = ds_reg.compute()

ds_reg


In [ ]:
import xarray as xr
import metpy.calc as mpcalc
from metpy.units import units

# --- load surface geopotential and take the SAME lon/lat selection strategy as ds ---
z_s = xr.open_dataset("/work/uc1275/u301827/02_MSE/surface_geopotential_zs.nc")

# If you're using a BOX-mean ds (recommended), define the box here too:
# box corners (0–360 lon)

# Make slices robust to coord order
lon_slice_z = slice(lon0, lon1) if z_s.longitude[0] < z_s.longitude[-1] else slice(lon1, lon0)
lat_slice_z = slice(lat0, lat1) if z_s.latitude[0]  < z_s.latitude[-1]  else slice(lat1, lat0)

# Regional mean surface geopotential (single scalar)
z_surface = (
    z_s["z"]
    .sel(longitude=lon_slice_z, latitude=lat_slice_z)
    .mean(dim=["longitude", "latitude"], skipna=True)
    #.isel(time=0)  # if time exists; remove this line if z has no time dim
    .item()
)

# --- Relative humidity from tasmax + 2d_at_tasmax (does NOT modify ds in-place) ---
t2m_units  = (ds_reg["tasmax"].data) * units.kelvin
td2m_units = (ds_reg["2d_at_tasmax"].data) * units.kelvin

rh = mpcalc.relative_humidity_from_dewpoint(t2m_units, td2m_units)  # dimensionless (0–1)

# attach RH as a normal float DataArray (0–1). Multiply by 100 if you want %.
rh_da = xr.DataArray(
    rh.magnitude,
    coords=ds_reg["tasmax"].coords,
    dims=ds_reg["tasmax"].dims,
    name="rh",
    attrs={"long_name": "Relative humidity", "units": "1"},
)

ds_reg = ds_reg.assign(rh=rh_da)

In [ ]:
import numpy as np
from scipy.special import lambertw
import xarray as xr

def compute_LCL(T, qv, Rh, z, g=9.81, p0 = 1000*100):
    """
    Compute the LCL temperature and height from temperature, humidity, and geopotential.
    
    Parameters
    ----------
    T : array-like or xarray.DataArray
        Temperature at reference level (K)
    qv : array-like or xarray.DataArray
        Specific humidity at reference level (kg/kg)
    Rh : array-like or xarray.DataArray
        Relative humidity at reference level (0 to 1)
    z : array-like or xarray.DataArray
        Geopotential at reference level (m^2/s^2), will be divided by g to get height (m)
    g : float, optional
        Gravitational acceleration (m/s²), default is 9.81
    p : float, optional
        pressure level of input (default 1000 hPa)

    Returns
    -------
    TLCL : same shape as inputs
        Temperature at LCL (K)
    zLCL : same shape as inputs
        Height of LCL (m)
    """

    # Physical constants
    Ttr = 273.16  # Triple point temperature (K)
    E0v = 2.3740e6  # Latent heat of vaporization at Ttr (J/kg)
    cvl = 4119      # Specific heat liquid water (J/kg/K)
    cvv = 1418      # Specific heat water vapor (J/kg/K)
    Rv  = 461       # Gas constant for water vapor (J/kg/K)
    cpv = cvv + Rv
    Ra  = 287.04    # Gas constant for dry air (J/kg/K)
    cva = 719
    cpa = cva + Ra

    # Moist air specific gas constant and heat capacity
    Rm  = (1 - qv) * Ra + qv * Rv
    cpm = (1 - qv) * cpa + qv * cpv

    # Coefficients for LCL temperature equation
    a = cpm / Rm + (cvl - cpv) / Rv
    b = - (E0v - Ttr * (cvv - cvl)) / (Rv * T)
    c = b / a

    # Argument to Lambert W function
    RHcec = c * np.exp(c) * Rh**(1 / a)

    # Compute TLCL using -1 branch of Lambert W
    TLCL = T * c / lambertw(RHcec, k=-1).real

    # Compute zLCL from TLCL
    z = z / g  # convert geopotential to height
    zLCL = z + (cpm / g) * (T - TLCL)
    zLCL_old = z+cpm/g * (T-55-1/(1/(T-55)-np.log(Rh)/2840))
    
    pLCL =  p0 * (TLCL / T) ** (cpm / Rm)

    return TLCL, zLCL, pLCL


In [ ]:
TLCLmax, zLCLmax, pLCLmax = compute_LCL(ds_reg.tasmax, ds_reg.q_at_tasmax, ds_reg.rh.values ,z_surface, p0 = ds_reg.sp.values)
ds_reg["pLCLmax"] = pLCLmax

In [ ]:
# Constants
R = 287.0       # J/kg/K, dry air gas constant
cp = 1004.0     # J/kg/K, dry air heat capacity
p_ref = 500e2   # 500 hPa in Pa

# Ensure surface pressure exists
ps = ds_reg["sp_at_tasmax"]      # surface pressure in Pa
T500 = ds_reg["t_at_tasmax"]# temperature at 500 hPa in K (replace with correct var)

# Compute theoretical upper bound
T_max_bound = T500 * (ps / p_ref)**(R / cp)


ds_reg["t_bound"] = T_max_bound

In [ ]:
import numpy as np
import xarray as xr

# -----------------------------------------------------------
# Input (ds is NOT altered anywhere below)
# -----------------------------------------------------------
tasmax = ds_reg["tasmax"].copy()

# -----------------------------------------------------------
# Helper: safe argmax that tolerates NaNs
# -----------------------------------------------------------
def safe_nanargmax(a, axis=0):
    a = np.asarray(a)
    all_nan = np.all(np.isnan(a), axis=axis)
    filled = np.where(np.isnan(a), -np.inf, a)
    idx = np.argmax(filled, axis=axis)
    if np.any(all_nan):
        idx = idx.astype(float)
        idx[all_nan] = np.nan
    return idx

# -----------------------------------------------------------
# 1) Index of TXx *within each year*
# -----------------------------------------------------------
txx_idx = (
    tasmax
    .groupby("time.year")
    .reduce(safe_nanargmax, dim="time")
    .dropna("year")
    .astype(int)
)

years = txx_idx["year"].values
idxs  = txx_idx.values

# -----------------------------------------------------------
# 2) Build composite WITHOUT modifying ds
# -----------------------------------------------------------
composite_list = []

for year, idx_in_year in zip(years, idxs):
    year = int(year)
    idx_in_year = int(idx_in_year)

    # yearly slice (temporary object)
    tas_year = tasmax.sel(time=str(year))

    # TXx timestamp for that year
    txx_time = tas_year.time.isel(time=idx_in_year).values

    # select corresponding timestep from ds (temporary object)
    comp = ds_reg.sel(time=txx_time, method="nearest")

    # store the matched timestamp
    actual_time = comp.time.values

    # build synthetic dimensions (only on comp)
    comp = comp.expand_dims(time=[0])
    comp = comp.assign_coords(time=("time", [0]))

    comp = comp.expand_dims(composite_year=[year])
    comp = comp.assign_coords(
        txx_time=("composite_year", [np.datetime64(actual_time)])
    )

    composite_list.append(comp)

# -----------------------------------------------------------
# 3) Concatenate safely (no coordinate alignment)
# -----------------------------------------------------------
composite_ds = xr.concat(
    composite_list,
    dim="composite_year",
    coords="minimal",
    compat="override",
    join="override",
)

composite_ds


In [ ]:
def pLCL_from_RH(Rh, T, qv, cpm, Rm, c, p0):
    """
    Compute pLCL from RH using Lambert-W formulation.
    Rh : relative humidity [0-1]
    T  : temperature at reference level [K]
    qv : specific humidity [kg/kg]
    cpm, Rm, c : precomputed coefficients from compute_LCL
    p0 : reference pressure (Pa)
    """
    from scipy.special import lambertw
    import numpy as np

    RHcec = c * np.exp(c) * Rh**(1 / a)
    TLCL = T * c / lambertw(RHcec, k=-1).real
    return p0 * (TLCL / T) ** (cpm / Rm)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np

plt.close("all")  # makes reruns clean

# keep datasets separate (DO NOT overwrite ds)
ds_era = ds_reg         # original big ERA5 point dataset
ds_comp = composite_ds  # TXx composite

# -------------------------------------------------------------
# 1. ERA5 JJA background (grey underlay)
# -------------------------------------------------------------
jja = ds_era.sel(time=ds_era.time.dt.month.isin([6, 7, 8]))

tasmax_era = jja["tasmax"] - 273.15
q_era = jja["t_bound"] - 273.15
mse_era = tasmax_era

x_era = mse_era.values.ravel()
y_era = q_era.values.ravel()
mask = ~np.isnan(x_era) & ~np.isnan(y_era)
x_era, y_era = x_era[mask], y_era[mask]

# -------------------------------------------------------------
# 2. Composite data (TXx day only => time index = 0)
# -------------------------------------------------------------
tasmax = ds_comp["tasmax"] - 273.15
q = ds_comp["t_bound"] - 273.15
mse = tasmax

if "lev" in q.dims:
    q = q.isel(lev=0)

day0_idx = 0

tas0 = tasmax.isel(time=day0_idx)
q0   = q.isel(time=day0_idx)
mse0 = mse.isel(time=day0_idx)

nyears = ds_comp.sizes["composite_year"]
tas_flat = tas0.values.reshape(nyears, -1)
q_flat   = q0.values.reshape(nyears, -1)
mse_flat = mse0.values.reshape(nyears, -1)

x = mse_flat.ravel()
y = q_flat.ravel()
valid = ~np.isnan(x) & ~np.isnan(y)
x, y = x[valid], y[valid]

# groups/colors per point
half = nyears // 2
groupA_year = np.arange(nyears) < half
groupA_point = np.repeat(groupA_year, tas_flat.shape[1])
groupA_point = groupA_point[valid.reshape(nyears, -1).ravel()]
colors = np.where(groupA_point, "tab:blue", "tab:orange")

# top 5 hottest years
tas_year_mean = np.nanmean(tas_flat, axis=1)
top5_year_idx = np.argsort(tas_year_mean)[-5:]

x_year = np.nanmean(mse_flat, axis=1)
y_year = np.nanmean(q_flat, axis=1)
year_colors = np.where(groupA_year, "tab:blue", "tab:orange")

# -------------------------------------------------------------
# 3. Plot
# -------------------------------------------------------------
fig = plt.figure(figsize=(10, 8))
gs = GridSpec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4],
              wspace=0.05, hspace=0.05)

ax_scatter = fig.add_subplot(gs[1, 0])
ax_histx   = fig.add_subplot(gs[0, 0], sharex=ax_scatter)
ax_histy   = fig.add_subplot(gs[1, 1], sharey=ax_scatter)

ax_scatter.scatter(y_era, x_era, c="grey", s=2, alpha=0.5, label="ERA5 JJA background")
ax_scatter.scatter(y, x, c=colors, s=20, marker="o", alpha=0.7, label="")
ax_scatter.scatter(y_year[top5_year_idx], x_year[top5_year_idx],
                   c=year_colors[top5_year_idx], edgecolor="black",
                   s=70, marker="D", label="Top 5 hottest years (TXx day)")

ax_scatter.set_ylabel("Tasmax [°C]")
ax_scatter.set_xlabel("Upper Dry Convective Bound [°C]")
ax_scatter.grid(True)
ax_scatter.legend()

ax_histx.hist(y_era, bins=30, color="grey", alpha=0.5, density=True)
ax_histx.hist(y, bins=10, color="tab:blue", alpha=0.35, density=True)
ax_histx.axis("off")

ax_histy.hist(x_era, bins=30, color="grey", alpha=0.5, density=True, orientation="horizontal")
ax_histy.hist(x, bins=10, color="tab:blue", alpha=0.35, density=True, orientation="horizontal")
ax_histy.axis("off")

fig.savefig(f"tasmax_plcl_composite_TXxday_q_axis_{region}.pdf", bbox_inches="tight")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

plt.close("all")

# ---- Inputs (assumed to already exist) ----
# ds_era: ERA5 point dataset with time as datetime64
# composite_ds: TXx composite with dims (composite_year, time=1, ...)

# ---- Prepare arrays needed for plotting only ----
# ERA5 JJA background
jja = ds_era.sel(time=ds_era.time.dt.month.isin([6, 7, 8]))
x_era = (jja["tasmax"] - 273.15).values.ravel()
y_era = (jja["t_bound"] - 273.15 - (jja["tasmax"] - 273.15)).values.ravel()
m = ~np.isnan(x_era) & ~np.isnan(y_era)
x_era, y_era = x_era[m], y_era[m]

# Composite (TXx day only)
tas0 = (composite_ds["tasmax"].isel(time=0) - 273.15)
q0   = (composite_ds["t_bound"].isel(time=0) - 273.15 - tas0)

ny = composite_ds.sizes["composite_year"]
x = tas0.values.reshape(ny, -1).ravel()
y = q0.values.reshape(ny, -1).ravel()
m = ~np.isnan(x) & ~np.isnan(y)
x, y = x[m], y[m]

# Top-5 hottest years (by spatial-mean tasmax on TXx day) + their mean points
tas_flat = tas0.values.reshape(ny, -1)
q_flat   = q0.values.reshape(ny, -1)
tas_year_mean = np.nanmean(tas_flat, axis=1)
top5 = np.argsort(tas_year_mean)[-5:]
x_year = np.nanmean(tas_flat, axis=1)
y_year = np.nanmean(q_flat, axis=1)

# ---- Plot (scatter + marginal histograms) ----
fig = plt.figure(figsize=(10, 8))
gs = GridSpec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4], wspace=0.05, hspace=0.05)

ax = fig.add_subplot(gs[1, 0])
ax_histx = fig.add_subplot(gs[0, 0], sharex=ax)
ax_histy = fig.add_subplot(gs[1, 1], sharey=ax)

ax.scatter(y_era, x_era, c="grey", s=2, alpha=0.5, label="ERA5 JJA background")
ax.scatter(y, x, s=20, alpha=0.7)
ax.scatter(y_year[top5], x_year[top5], s=80, marker="D", edgecolor="black", label="Top 5 hottest years")

ax.set_xlabel("Upper Dry Convective Bound - tasmax [°C]")
ax.set_ylabel("Tasmax [°C]")
ax.grid(True)
ax.legend()

ax_histx.hist(y_era, bins=30, color="grey", alpha=0.5, density=True)
ax_histx.hist(y, bins=10, alpha=0.35, density=True)
ax_histx.axis("off")

ax_histy.hist(x_era, bins=30, color="grey", alpha=0.5, density=True, orientation="horizontal")
ax_histy.hist(x, bins=10, alpha=0.35, density=True, orientation="horizontal")
ax_histy.axis("off")

fig.savefig(f"tasmax_tbound_composite_TXxday_q_axis_{region}.pdf", bbox_inches="tight")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

plt.close("all")

# ---- Inputs assumed to exist ----
# ds_era: ERA5 point dataset (datetime time)
# composite_ds: TXx composite (composite_year, time=1, ...)

# ---- ERA5 JJA background (only what is needed for plotting) ----
jja = ds_era.sel(time=ds_era.time.dt.month.isin([6, 7, 8]))
x_era = (jja["rh"] * 100).values.ravel()      # RH [%]
y_era = (jja["pLCLmax"]).values.ravel()          # pLCL [Pa]

m = ~np.isnan(x_era) & ~np.isnan(y_era)
x_era, y_era = x_era[m], y_era[m]

# ---- Composite (TXx day only => time=0) ----
rh0   = (composite_ds["rh"].isel(time=0) * 100)     # RH [%]
plcl0 = (composite_ds["pLCLmax"].isel(time=0))         # pLCL [Pa]

ny = composite_ds.sizes["composite_year"]
x = rh0.values.reshape(ny, -1).ravel()
y = plcl0.values.reshape(ny, -1).ravel()

m = ~np.isnan(x) & ~np.isnan(y)
x, y = x[m], y[m]

# Top-5 hottest years (by spatial-mean tasmax on TXx day) + their mean points
tas0 = (composite_ds["tasmax"].isel(time=0) - 273.15).values.reshape(ny, -1)
tas_year_mean = np.nanmean(tas0, axis=1)
top5 = np.argsort(tas_year_mean)[-5:]

x_year = np.nanmean(rh0.values.reshape(ny, -1), axis=1)
y_year = np.nanmean(plcl0.values.reshape(ny, -1), axis=1)

# ---- Empirical fits (log-log) ----
def loglog_fit(x_rh_percent, y_plcl_pa):
    m = (x_rh_percent > 0) & (y_plcl_pa > 0) & np.isfinite(x_rh_percent) & np.isfinite(y_plcl_pa)
    xr = (x_rh_percent[m] / 100.0)
    yr = y_plcl_pa[m]
    coeffs = np.polyfit(np.log(xr), np.log(yr), 1)
    beta = coeffs[0]
    alpha = np.exp(coeffs[1])
    return alpha, beta

alpha_era, beta_era = loglog_fit(x_era, y_era)
alpha_comp, beta_comp = loglog_fit(x, y)

RH_fit = np.linspace(5, 99, 200)  # RH in %
pLCL_fit_era  = alpha_era  * (RH_fit/100.0) ** beta_era
pLCL_fit_comp = alpha_comp * (RH_fit/100.0) ** beta_comp

# ---- Plot ----
fig = plt.figure(figsize=(10, 8))
gs = GridSpec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4], wspace=0.05, hspace=0.05)

ax = fig.add_subplot(gs[1, 0])
ax_histx = fig.add_subplot(gs[0, 0], sharex=ax)
ax_histy = fig.add_subplot(gs[1, 1], sharey=ax)

ax.scatter(y_era, x_era, c="grey", s=2, alpha=0.5, label="ERA5 JJA background")
ax.scatter(y, x, s=20, alpha=0.7, label="")
ax.scatter(y_year[top5], x_year[top5], s=80, marker="D", edgecolor="black", label="Top 5 hottest years")

ax.plot(pLCL_fit_era, RH_fit, "k--", lw=2, alpha=0.5, label="Empirical fit (ERA5)")
ax.plot(pLCL_fit_comp, RH_fit, "r--", lw=2, alpha=0.5, label="Empirical fit (TXx composite)")

ax.set_xlabel("Lifting condensation level [Pa]")
ax.set_ylabel("RH [%]")
ax.grid(True)
ax.legend()

ax_histx.hist(y_era, bins=30, color="grey", alpha=0.5, density=True)
ax_histx.hist(y, bins=10, alpha=0.35, density=True)
ax_histx.axis("off")

ax_histy.hist(x_era, bins=30, color="grey", alpha=0.5, density=True, orientation="horizontal")
ax_histy.hist(x, bins=10, alpha=0.35, density=True, orientation="horizontal")
ax_histy.axis("off")

fig.savefig(f"rh_plcl_composite_TXxday_q_axis_{region}.pdf", bbox_inches="tight")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

plt.close("all")

# ---- Inputs (assumed to already exist) ----
# ds_era: ERA5 point dataset with time as datetime64
# composite_ds: TXx composite with dims (composite_year, time=1, ...)

# ---- Prepare arrays needed for plotting only ----
# ERA5 JJA background
jja = ds_era.sel(time=ds_era.time.dt.month.isin([6, 7, 8]))
x_era = (jja["tasmax"] - 273.15).values.ravel()
y_era = (jja["blh_at_tasmax"]).values.ravel()
m = ~np.isnan(x_era) & ~np.isnan(y_era)
x_era, y_era = x_era[m], y_era[m]

# Composite (TXx day only)
tas0 = (composite_ds["tasmax"].isel(time=0) - 273.15)
q0   = (composite_ds["blh_at_tasmax"].isel(time=0))

ny = composite_ds.sizes["composite_year"]
x = tas0.values.reshape(ny, -1).ravel()
y = q0.values.reshape(ny, -1).ravel()
m = ~np.isnan(x) & ~np.isnan(y)
x, y = x[m], y[m]

# Top-5 hottest years (by spatial-mean tasmax on TXx day) + their mean points
tas_flat = tas0.values.reshape(ny, -1)
q_flat   = q0.values.reshape(ny, -1)
tas_year_mean = np.nanmean(tas_flat, axis=1)
top5 = np.argsort(tas_year_mean)[-5:]
x_year = np.nanmean(tas_flat, axis=1)
y_year = np.nanmean(q_flat, axis=1)

# ---- Plot (scatter + marginal histograms) ----
fig = plt.figure(figsize=(10, 8))
gs = GridSpec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4], wspace=0.05, hspace=0.05)

ax = fig.add_subplot(gs[1, 0])
ax_histx = fig.add_subplot(gs[0, 0], sharex=ax)
ax_histy = fig.add_subplot(gs[1, 1], sharey=ax)

ax.scatter(y_era, x_era, c="grey", s=2, alpha=0.5, label="ERA5 JJA background")
ax.scatter(y, x, s=20, alpha=0.7)
ax.scatter(y_year[top5], x_year[top5], s=80, marker="D", edgecolor="black", label="Top 5 hottest years")

ax.set_xlabel("Boundary Layer Height at Tasmax [m]")
ax.set_ylabel("Tasmax [°C]")
ax.grid(True)
ax.legend()

ax_histx.hist(y_era, bins=30, color="grey", alpha=0.5, density=True)
ax_histx.hist(y, bins=10, alpha=0.35, density=True)
ax_histx.axis("off")

ax_histy.hist(x_era, bins=30, color="grey", alpha=0.5, density=True, orientation="horizontal")
ax_histy.hist(x, bins=10, alpha=0.35, density=True, orientation="horizontal")
ax_histy.axis("off")

fig.savefig(f"tasmax_blh_tasmax_composite_TXxday_q_axis_{region}.pdf", bbox_inches="tight")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

plt.close("all")

# ---- Inputs (assumed to already exist) ----
# ds_era: ERA5 point dataset with time as datetime64
# composite_ds: TXx composite with dims (composite_year, time=1, ...)

# ---- Prepare arrays needed for plotting only ----
# ERA5 JJA background
jja = ds_era.sel(time=ds_era.time.dt.month.isin([6, 7, 8]))
x_era = (jja["tasmax"] - 273.15).values.ravel()
y_era = (jja["blh"]).values.ravel()
m = ~np.isnan(x_era) & ~np.isnan(y_era)
x_era, y_era = x_era[m], y_era[m]

# Composite (TXx day only)
tas0 = (composite_ds["tasmax"].isel(time=0) - 273.15)
q0   = (composite_ds["blh"].isel(time=0))

ny = composite_ds.sizes["composite_year"]
x = tas0.values.reshape(ny, -1).ravel()
y = q0.values.reshape(ny, -1).ravel()
m = ~np.isnan(x) & ~np.isnan(y)
x, y = x[m], y[m]

# Top-5 hottest years (by spatial-mean tasmax on TXx day) + their mean points
tas_flat = tas0.values.reshape(ny, -1)
q_flat   = q0.values.reshape(ny, -1)
tas_year_mean = np.nanmean(tas_flat, axis=1)
top5 = np.argsort(tas_year_mean)[-5:]
x_year = np.nanmean(tas_flat, axis=1)
y_year = np.nanmean(q_flat, axis=1)

# ---- Plot (scatter + marginal histograms) ----
fig = plt.figure(figsize=(10, 8))
gs = GridSpec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4], wspace=0.05, hspace=0.05)

ax = fig.add_subplot(gs[1, 0])
ax_histx = fig.add_subplot(gs[0, 0], sharex=ax)
ax_histy = fig.add_subplot(gs[1, 1], sharey=ax)

ax.scatter(y_era, x_era, c="grey", s=2, alpha=0.5, label="ERA5 JJA background")
ax.scatter(y, x, s=20, alpha=0.7)
ax.scatter(y_year[top5], x_year[top5], s=80, marker="D", edgecolor="black", label="Top 5 hottest years")

ax.set_xlabel("Daily Mean Boundary Layer Height [m]")
ax.set_ylabel("Tasmax [°C]")
ax.grid(True)
ax.legend()

ax_histx.hist(y_era, bins=30, color="grey", alpha=0.5, density=True)
ax_histx.hist(y, bins=10, alpha=0.35, density=True)
ax_histx.axis("off")

ax_histy.hist(x_era, bins=30, color="grey", alpha=0.5, density=True, orientation="horizontal")
ax_histy.hist(x, bins=10, alpha=0.35, density=True, orientation="horizontal")
ax_histy.axis("off")

fig.savefig(f"tasmax_blh_composite_TXxday_q_axis_{region}.pdf", bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np

plt.close("all")

# keep datasets separate
ds_era = ds_reg
ds_comp = composite_ds

# -------------------------------------------------------------
# 1. ERA5 JJA background
# -------------------------------------------------------------
jja = ds_era.sel(time=ds_era.time.dt.month.isin([6, 7, 8]))

tasmax_era = jja["tasmax"] - 273.15
q_era = jja["t_bound"] - 273.15

# SWITCH axes
x_era = tasmax_era.values.ravel()
y_era = q_era.values.ravel()

mask = ~np.isnan(x_era) & ~np.isnan(y_era)
x_era, y_era = x_era[mask], y_era[mask]

# -------------------------------------------------------------
# 2. Composite data
# -------------------------------------------------------------
tasmax = ds_comp["tasmax"] - 273.15
q = ds_comp["t_bound"] - 273.15

if "lev" in q.dims:
    q = q.isel(lev=0)

tas0 = tasmax.isel(time=0)
q0   = q.isel(time=0)

nyears = ds_comp.sizes["composite_year"]
tas_flat = tas0.values.reshape(nyears, -1)
q_flat   = q0.values.reshape(nyears, -1)

# SWITCH axes
x = tas_flat.ravel()
y = q_flat.ravel()

valid = ~np.isnan(x) & ~np.isnan(y)
x, y = x[valid], y[valid]

# -------------------------------------------------------------
# Top 5 hottest years
# -------------------------------------------------------------
tas_year_mean = np.nanmean(tas_flat, axis=1)
top5_year_idx = np.argsort(tas_year_mean)[-5:]

x_year = np.nanmean(tas_flat, axis=1)
y_year = np.nanmean(q_flat, axis=1)

# -------------------------------------------------------------
# Plot
# -------------------------------------------------------------
fig = plt.figure(figsize=(10, 8))
gs = GridSpec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4],
              wspace=0.05, hspace=0.05)

ax_scatter = fig.add_subplot(gs[1, 0])
ax_histx   = fig.add_subplot(gs[0, 0], sharex=ax_scatter)
ax_histy   = fig.add_subplot(gs[1, 1], sharey=ax_scatter)

ax_scatter.scatter(x_era, y_era,
                   c="grey", s=2, alpha=0.5,
                   label="ERA5 JJA background")

ax_scatter.scatter(x, y,
                   c="tab:blue", s=20, alpha=0.7,
                   label="TXx")

ax_scatter.scatter(x_year[top5_year_idx], y_year[top5_year_idx],
                   c="black", edgecolor="black",
                   s=70, marker="D",
                   label="Top 5 hottest years")

ax_scatter.set_xlabel("Daily Maximum Near Surface Temperature [°C]")
ax_scatter.set_ylabel("Upper Dry Convective Bound [°C]")
ax_scatter.grid(True)
ax_scatter.legend()

# Histograms (also switched)
ax_histx.hist(x_era, bins=30, color="grey", alpha=0.5, density=True)
ax_histx.hist(x, bins=10, color="tab:blue", alpha=0.35, density=True)
ax_histx.axis("off")

ax_histy.hist(y_era, bins=30, color="grey", alpha=0.5, density=True, orientation="horizontal")
ax_histy.hist(y, bins=10, color="tab:blue", alpha=0.35, density=True, orientation="horizontal")
ax_histy.axis("off")

fig.savefig(f"tasmax_plcl_composite_TXxday_q_axis_{region}_switched.pdf", bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np

plt.close("all")

# keep datasets separate
ds_era = ds_reg
ds_comp = composite_ds

# -------------------------------------------------------------
# 1. ERA5 JJA background
# -------------------------------------------------------------
jja = ds_era.sel(time=ds_era.time.dt.month.isin([6, 7, 8]))

tasmax_era = jja["tasmax"] - 273.15
q_era = jja["q"]*1e3

# SWITCH axes
x_era = tasmax_era.values.ravel()
y_era = q_era.values.ravel()

mask = ~np.isnan(x_era) & ~np.isnan(y_era)
x_era, y_era = x_era[mask], y_era[mask]

# -------------------------------------------------------------
# 2. Composite data
# -------------------------------------------------------------
tasmax = ds_comp["tasmax"] - 273.15
q = ds_comp["q_at_tasmax"]*1e3

if "lev" in q.dims:
    q = q.isel(lev=0)

tas0 = tasmax.isel(time=0)
q0   = q.isel(time=0)

nyears = ds_comp.sizes["composite_year"]
tas_flat = tas0.values.reshape(nyears, -1)
q_flat   = q0.values.reshape(nyears, -1)

# SWITCH axes
x = tas_flat.ravel()
y = q_flat.ravel()

valid = ~np.isnan(x) & ~np.isnan(y)
x, y = x[valid], y[valid]

# -------------------------------------------------------------
# Top 5 hottest years
# -------------------------------------------------------------
tas_year_mean = np.nanmean(tas_flat, axis=1)
top5_year_idx = np.argsort(tas_year_mean)[-5:]

x_year = np.nanmean(tas_flat, axis=1)
y_year = np.nanmean(q_flat, axis=1)

# -------------------------------------------------------------
# Plot
# -------------------------------------------------------------
fig = plt.figure(figsize=(10, 8))
gs = GridSpec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4],
              wspace=0.05, hspace=0.05)

ax_scatter = fig.add_subplot(gs[1, 0])
ax_histx   = fig.add_subplot(gs[0, 0], sharex=ax_scatter)
ax_histy   = fig.add_subplot(gs[1, 1], sharey=ax_scatter)

ax_scatter.scatter(x_era, y_era,
                   c="grey", s=2, alpha=0.5,
                   label="ERA5 JJA background")

ax_scatter.scatter(x, y,
                   c="tab:blue", s=20, alpha=0.7,
                   label="TXx")

ax_scatter.scatter(x_year[top5_year_idx], y_year[top5_year_idx],
                   c="black", edgecolor="black",
                   s=70, marker="D",
                   label="Top 5 hottest years")

ax_scatter.set_xlabel("Daily Maximum Near Surface Temperature [°C]")
ax_scatter.set_ylabel("Near Surface Moisture [g/kg]")
ax_scatter.grid(True)
ax_scatter.legend()

# Histograms (also switched)
ax_histx.hist(x_era, bins=30, color="grey", alpha=0.5, density=True)
ax_histx.hist(x, bins=10, color="tab:blue", alpha=0.35, density=True)
ax_histx.axis("off")

ax_histy.hist(y_era, bins=30, color="grey", alpha=0.5, density=True, orientation="horizontal")
ax_histy.hist(y, bins=10, color="tab:blue", alpha=0.35, density=True, orientation="horizontal")
ax_histy.axis("off")

fig.savefig(f"tasmax_q_composite_TXxday_q_axis_{region}_switched.pdf", bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np

plt.close("all")

# keep datasets separate
ds_era = ds_reg
ds_comp = composite_ds

# -------------------------------------------------------------
# 1. ERA5 JJA background
# -------------------------------------------------------------
jja = ds_era.sel(time=ds_era.time.dt.month.isin([6, 7, 8]))

tasmax_era = jja["tasmax"] - 273.15
q_era = jja["t_bound"] - 273.15

# SWITCH axes
x_era = tasmax_era.values.ravel()
y_era = q_era.values.ravel()

mask = ~np.isnan(x_era) & ~np.isnan(y_era)
x_era, y_era = x_era[mask], y_era[mask]

# -------------------------------------------------------------
# 2. Composite data
# -------------------------------------------------------------
tasmax = ds_comp["tasmax"] - 273.15
q = ds_comp["t_bound"] - 273.15

if "lev" in q.dims:
    q = q.isel(lev=0)

tas0 = tasmax.isel(time=0)
q0   = q.isel(time=0)

nyears = ds_comp.sizes["composite_year"]
tas_flat = tas0.values.reshape(nyears, -1)
q_flat   = q0.values.reshape(nyears, -1)

# SWITCH axes
x = tas_flat.ravel()
y = q_flat.ravel()

valid = ~np.isnan(x) & ~np.isnan(y)
x, y = x[valid], y[valid]

# -------------------------------------------------------------
# Top 5 hottest years
# -------------------------------------------------------------
tas_year_mean = np.nanmean(tas_flat, axis=1)
top5_year_idx = np.argsort(tas_year_mean)[-5:]

x_year = np.nanmean(tas_flat, axis=1)
y_year = np.nanmean(q_flat, axis=1)

# -------------------------------------------------------------
# Plot
# -------------------------------------------------------------
fig = plt.figure(figsize=(10, 8))
gs = GridSpec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4],
              wspace=0.05, hspace=0.05)

ax_scatter = fig.add_subplot(gs[1, 0])
ax_histx   = fig.add_subplot(gs[0, 0], sharex=ax_scatter)
ax_histy   = fig.add_subplot(gs[1, 1], sharey=ax_scatter)

ax_scatter.scatter(x_era, y_era,
                   c="grey", s=2, alpha=0.5,
                   label="ERA5 JJA background")

ax_scatter.scatter(x, y,
                   c="tab:blue", s=20, alpha=0.7,
                   label="TXx")

ax_scatter.scatter(x_year[top5_year_idx], y_year[top5_year_idx],
                   c="black", edgecolor="black",
                   s=70, marker="D",
                   label="Top 5 hottest years")

ax_scatter.set_xlabel("Daily Maximum Near Surface Temperature [°C]")
ax_scatter.set_ylabel("Upper Dry Convective Bound [°C]")
ax_scatter.grid(True)
ax_scatter.legend()

# Histograms (also switched)
ax_histx.hist(x_era, bins=30, color="grey", alpha=0.5, density=True)
ax_histx.hist(x, bins=10, color="tab:blue", alpha=0.35, density=True)
ax_histx.axis("off")

ax_histy.hist(y_era, bins=30, color="grey", alpha=0.5, density=True, orientation="horizontal")
ax_histy.hist(y, bins=10, color="tab:blue", alpha=0.35, density=True, orientation="horizontal")
ax_histy.axis("off")

fig.savefig(f"tasmax_plcl_composite_TXxday_q_axis_{region}_switched.pdf", bbox_inches="tight")
plt.show()
